In [ ]:
from ftplib import FTP, error_perm, error_temp
from urllib.error import URLError
import urllib.request
import boto3
from botocore.exceptions import ClientError

In [5]:
s3_client = boto3.client('s3')

In [2]:
# --------------------
# FUNÇÃO
# --------------------
def mapear_arquivos_ftp() -> dict:
    '''
    Cria um dicionário contendo a competência, no formato aamm, dos arquivos mais recentes disponibilizados no módulo 
    Profissionais do CNES, no servidor FTP do DataSUS, bem como uma lista com os nomes dos arquivos dessa competência.

    Returns:
        dict_arquivos(dict): Dicionário com a competência mais recente e a lista de arquivos. 
    '''
    # Dicionário que armazenará o resultado da função
    dict_arquivos = {}

    # Variáveis de host e diretório
    host = 'ftp.datasus.gov.br'
    dir_pf = 'dissemin/publicos/CNES/200508_/Dados/PF/'

    print(f'Iniciando conexão com o servidor {host}')

    try:
        with FTP(host) as servidor:
            
            # Acessa o servidor FTP do DataSUS como usuário anônimo
            servidor.login()
            print('Conexão estabelecida.')

            # Navega até o diretório que contém as bases de dados
            servidor.cwd(dir_pf)
            print(f'Navegando para o diretório {dir_pf}')
            
            # Armazena no dicionário a competência atual dos arquivos do servidor FTP (formato: aamm)
            dict_arquivos['competencia'] = int(servidor.nlst()[-1][-8:-4])
            print(f'Competência atual (FTP): {dict_arquivos['competencia']}')

            # Cria a string de filtragem dos arquivos
            filtro = '*' + str(dict_arquivos['competencia']) + '.dbc'
            print(f'String de filtro criada: {filtro}')

            # Armazena no dicionário a lista com os arquivos da competência mais atual do servidor FTP
            dict_arquivos['arquivos'] = servidor.nlst(filtro)

        return dict_arquivos

    except error_perm as e:
        # Logs de erros de caráter permanente (inexistência de diretórios e/ou arquivos)
        print(f'Falha permanente no servidor FTP: {e}')
        return None

    except error_temp as e:
        # Logs de erros de caráter temporário (instabilidade no servidor)
        print(f'Falha temporária no servidor FTP: {e}')        
        return None

    except Exception as e:
        # Logs de error gerais, armazenando a mensagem completa do erro.
        print(f'Erro: {e}')   
        return None


# --------------------
# EXECUÇÃO
# --------------------
resposta_funcao = mapear_arquivos_ftp()
print(f'Retorno da função: {resposta_funcao}')

Iniciando conexão com o servidor ftp.datasus.gov.br
Conexão estabelecida.
Navegando para o diretório dissemin/publicos/CNES/200508_/Dados/PF/
Competência atual (FTP): 2603
String de filtro criada: *2603.dbc
Retorno da função: {'competencia': 2603, 'arquivos': ['PFAC2603.dbc', 'PFAL2603.dbc', 'PFAM2603.dbc', 'PFAP2603.dbc', 'PFBA2603.dbc', 'PFCE2603.dbc', 'PFDF2603.dbc', 'PFES2603.dbc', 'PFGO2603.dbc', 'PFMA2603.dbc', 'PFMG2603.dbc', 'PFMS2603.dbc', 'PFMT2603.dbc', 'PFPA2603.dbc', 'PFPB2603.dbc', 'PFPE2603.dbc', 'PFPI2603.dbc', 'PFPR2603.dbc', 'PFRJ2603.dbc', 'PFRN2603.dbc', 'PFRO2603.dbc', 'PFRR2603.dbc', 'PFRS2603.dbc', 'PFSC2603.dbc', 'PFSE2603.dbc', 'PFSP2603.dbc', 'PFTO2603.dbc']}


In [6]:
# --------------------
# MOCK DE DADOS
# --------------------
bucket = 'pipesus'


# --------------------
# FUNÇÃO
# --------------------
def mapear_arquivos_bucket(bucket:str) -> dict:
    '''
    Cria um dicionário contendo a competência, no formato aamm, dos arquivos mais recentes disponibilizados no bucket 
    do Amazon S3, bem como uma lista com o nome dos arquivos dessa competência.

    Args:
        bucket(str): Nome do bucket de destino no Amazon S3.

    Returns:
        dict_arquivos(dict): Dicionário com a competência mais recente e a lista de arquivos. 
    '''
    
    print(f'Nome do Bucket: {bucket}')
    dict_arquivos = {}

    # Consulta os arquivos existentes na camada bronze do bucket
    resposta_s3 = s3_client.list_objects_v2(Bucket=bucket, Prefix='bronze/cnes/profissionais/')
    print(f'Resposta Amazon: {resposta_s3}')
    
    if 'Contents' in resposta_s3:
        
        # Cria uma lista contendo os nomes dos arquivos da camada bronze
        arquivos_bucket = [arquivo['Key'] for arquivo in resposta_s3['Contents']]

        # Identifica a competência dos arquivos na camada bronze
        competencia = arquivos_bucket[-1][-8:-4]

        # Insere a competência e a lista com o nome dos arquivos no dicionário
        dict_arquivos['competencia'] = int(competencia)
        dict_arquivos['arquivos'] = arquivos_bucket
    
    else:
        dict_arquivos['competencia'] = 0
        dict_arquivos['arquivos'] = []

    return dict_arquivos


# --------------------
# EXECUÇÃO
# --------------------
resposta_funcao = mapear_arquivos_bucket(bucket)
print(f'Retorno da função: {resposta_funcao}')

Nome do Bucket: pipesus
Resposta Amazon: {'ResponseMetadata': {'RequestId': 'TXC79707B7869ZCG', 'HostId': 'Br8WRRdz8UMjFOQcOiqse9WdmpVUJtW8ds4BHHYW2yZY2Ci2jARPG4xMiJdjQuj3xLDB803cYgEYaahGuPq9+NnQLeuYLWJr', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'Br8WRRdz8UMjFOQcOiqse9WdmpVUJtW8ds4BHHYW2yZY2Ci2jARPG4xMiJdjQuj3xLDB803cYgEYaahGuPq9+NnQLeuYLWJr', 'x-amz-request-id': 'TXC79707B7869ZCG', 'date': 'Tue, 12 May 2026 11:48:01 GMT', 'x-amz-bucket-region': 'us-east-1', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'IsTruncated': False, 'Contents': [{'Key': 'bronze/cnes/profissionais/PFAC2603.dbc', 'LastModified': datetime.datetime(2026, 5, 12, 6, 54, 54, tzinfo=tzutc()), 'ETag': '"006b4ef3352592de4e14c8caa6d0da9b"', 'ChecksumAlgorithm': ['CRC32'], 'ChecksumType': 'FULL_OBJECT', 'Size': 1244524, 'StorageClass': 'STANDARD'}], 'Name': 'pipesus', 'Prefix': 'bronze/cnes/profissionais/', 'MaxKeys': 1000, 'EncodingType': 'url'

In [15]:
def transferir_ftp_para_s3(arquivos:list, bucket:str):
    '''
    Transfere os arquivos do CNES Profissionais do servidor FTP do DataSUS para a camada bronze do
    bucket no Amazon S3.

    Args:
        arquivos(list): Lista contendo o nome dos arquivos que serão baixados.
        bucket(str): Nome do bucket de destino no Amazon S3.
    '''

    print(f'Nome do Bucket: {bucket}')
    print(f'Arquivos para baixar: {arquivos}')

    # URL base do servidor FTP
    url_ftp = 'ftp://ftp.datasus.gov.br/dissemin/publicos/CNES/200508_/Dados/PF'

    # Contabiliza a quantidade de arquivos baixados
    cont_downloads = 0

    # Cria uma lista de arquivos que não tiveram a transferência concluída para o S3
    arquivos_erro = []

    # Realiza o download de cada arquivo da lista arquivos
    for arquivo in arquivos:

        try:
            # Cria uma requisição HTTPS para baixar o arquivo
            with urllib.request.urlopen(f'{url_ftp}/{arquivo}') as arquivo_path:

                print(f'Baixando arquivo: {arquivo}')

                # Nome do arquivo que será armazenado na camada bronze no bucket
                nome_objeto = f'bronze/cnes/profissionais/{arquivo}'

                # Faz o upload do arquivo no bucket do Amazon S3
                s3_client.upload_fileobj(
                    Fileobj=arquivo_path,
                    Bucket=bucket,
                    Key=nome_objeto
                )
                print(f'Download concluído: {arquivo}')
                
            # Incrementa a quantidade de downloas realizados
            cont_downloads += 1
                
        except URLError as e:
            # Inclui o nome do arquivo na lista de arquivos que deram erro
            arquivos_erro.append(arquivo)
            print(f'Não foi possível baixar o arquivo {arquivo}: {e}')
            continue

    print(f'Transferência concluída. Arquivos baixados: {cont_downloads}/{len(arquivos)}.')

    if cont_downloads != len(arquivos):
        print(f'Arquivos não baixados: {arquivos_erro}.')

In [ ]:
def excluir_arquivos_bucket(arquivos:list, bucket:str):
    '''
    Exclui os arquivos do bucket no Amazon S3.

    Args:
        arquivos(list): Lista de arquivos a serem excluídos do bucket. 
        bucket(str): Nome do bucket de destino no Amazon S3.
    '''

    print(f'Nome do bucket: {bucket}')
    print(f'Arquivos para deletar: {arquivos}')

    # Cria uma lista de dicionários dos arquivos a serem excluídos
    arquivos_deletar = [{'Key': arquivo} for arquivo in arquivos]

    try:
        # Exclui os arquivos do bucket
        s3_client.delete_objects(
            Bucket=bucket,
            Delete={
                'Objects': arquivos_deletar,
                'Quiet': True
            }
        )
        print(f'Arquivos excluídos com sucesso.')

    except ClientError as e:
        print(f'Erro ao tentar excluir os arquivos: {e}')

{
    'version': '0', 
    'id': 'ce6a033b-fc53-44a3-b45c-7d0bc2c2902f', 
    'detail-type': 'Scheduled Event', 
    'source': 'aws.scheduler', 
    'account': '438641189047', 
    'time': '2026-05-12T14:41:00Z', 
    'region': 'us-east-1', 
    'resources': ['arn:aws:scheduler:us-east-1:438641189047:schedule/default/pipesus-agendamento-diario'], 
    'detail': '{}'
}